---
# BYOL Implementation
---

In this exercise, we implement **BYOL**: *Bootstrap Your Own Latent*. BYOL is a self-supervised method that learns image representations from two augmented views of the same CIFAR-10 image.

<span style="color:#E74C3C">Note: Since you have more implementation freedom, the tests might not cover everything correctly. If you think your solution is fine, then just continue.</span>

You may copy and adapt code from earlier exercises, especially:

- the CIFAR-10 data loading code,
- the CIFAR-style ResNet encoder from the ResNet exercise,
- the supervised training loop from previous CIFAR-10 exercises.

Paper: [Bootstrap Your Own Latent: A New Approach to Self-Supervised Learning](https://arxiv.org/abs/2006.07733)


---
# Imports
---

Run the code cell below after you have set up everything correctly.


In [1]:
import importlib
import torch

import data
import downstream
import models
import pretrain
import tests
import visual

---
## How BYOL Works
---

For each image, BYOL creates two independently augmented views. One direction of the computation is:

```text
image
 ├── augmentation 1 -> x1 -> online encoder -> online projector -> online predictor -> p1
 └── augmentation 2 -> x2 -> target encoder -> target projector                     -> z2
```

The loss makes the online prediction `p1` similar to the target projection `z2`. BYOL also applies the same idea in the opposite direction: `p2` is matched to `z1`.

Important details:

- the online branch is optimized by gradient descent,
- the target branch is not optimized by gradient descent,
- the target branch follows the online branch via exponential moving average,
- the predictor exists only in the online branch,
- the target projection is detached from the computational graph,
- no negative image pairs are used.


---
## Two Augmented Views
---

BYOL trains on two differently augmented views of the same image.

We use a CIFAR-10 adaptation of the augmentation recipe from the BYOL paper.  
The original recipe was designed for ImageNet images of size `224 × 224`; here we resize crops to `32 × 32` and use a smaller Gaussian blur kernel.

## **Task:**

Implement `TwoViewTransform` in `data.py`.

Use two augmentation pipelines, `transform_1` and `transform_2`.

Both pipelines should use:

- `RandomResizedCrop(32, scale=(0.08, 1.0), ratio=(3/4, 4/3))`
- `RandomHorizontalFlip(p=0.5)`
- `RandomApply([ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8)`
- `RandomGrayscale(p=0.2)`
- `RandomApply([GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=...)`
- `RandomSolarize(threshold=0.5, p=...)`
- `ToImage()`
- `ToDtype(torch.float32, scale=True)`
- `Normalize(CIFAR10_MEAN, CIFAR10_STD)`

The two pipelines should differ in the last two probabilities:

- `transform_1`: Gaussian blur probability `1.0`, solarization probability `0.0`
- `transform_2`: Gaussian blur probability `0.1`, solarization probability `0.2`

**Hint:** Use separate `ColorJitter` and `GaussianBlur` objects for the two transform pipelines. This avoids unintentionally sharing transform state.

In [2]:
importlib.reload(data)
tests.test_two_view_transform()

✅ PASS: TwoViewTransform looks correct.


True

---
## Visualizing the Two Views
---

The top row shows the first view of several images. The bottom row shows the corresponding second views. Images in the same column come from the same original CIFAR-10 image.

The two views should still show the same underlying image, but they should differ enough that the model cannot solve the task by copying pixels.


In [3]:
importlib.reload(data)
importlib.reload(visual)

byol_loader = data.make_byol_loader(batch_size=8, num_workers=0)
views, _ = next(iter(byol_loader))
view1, view2 = views

visual.two_view_examples(view1, view2).show()

---
## BYOL Loss
---

BYOL compares the online prediction with the target projection using cosine similarity:

$$
\operatorname{cos}(p, z) =
\frac{p^\top z}{\lVert p \rVert_2 \lVert z \rVert_2}
$$

The loss used here is:

$$
\mathcal{L}(p,z) = 2 - 2 \cdot \operatorname{cos}(p, \operatorname{sg}(z)),
$$

where $\operatorname{sg}$ means stop-gradient.

## **Task:**

Implement `negative_cosine_similarity` in `pretrain.py`.

Do not use `torch.nn.functional.cosine_similarity`. Implement the formula manually using normalization and a dot product.

**Hints:**
- Use `tensor.detach()` to stop gradients.
- You may use `torch.nn.functional.normalize`, or `F.normalize`, to normalize vectors.

In [4]:
importlib.reload(pretrain)
tests.test_negative_cosine_similarity()

✅ PASS: Negative cosine similarity is correct.


True

---
## Exponential Moving Average
---

The target network is updated using an exponential moving average:

```python
target = tau * target + (1 - tau) * online
```

Typical values are around `tau = 0.99`.

## **Task:**

Implement `update_moving_average` in `pretrain.py`.

**Hints:**

- Use in-place operations to change the target network weights.
- Perform the update in a context where PyTorch does not track gradients.
- Use `.parameters()` and `.buffers()` to access a module's parameters and state tensors.
- Since the online and target modules have the same structure, `zip(...)` can pair corresponding tensors.
- Buffers are non-trainable state tensors. BatchNorm running means and variances are common examples.
- Use `torch.is_floating_point(tensor)` for buffers: EMA-update floating-point buffers, and copy non-floating-point buffers.

In [5]:
importlib.reload(pretrain)
tests.test_ema_update()

✅ PASS: EMA target update is correct.


True

---
## BYOL Models
---

## **Task:**

Implement `BYOL` in `models.py`.

For the encoder, copy or adapt the CIFAR-style ResNet encoder from the previous ResNet exercise.

The BYOL class should contain these attributes:

- `online_encoder`: the ResNet encoder,
- `online_projector`: an MLP,
- `online_predictor`: an MLP,
- `target_encoder`: copy of the online encoder, with frozen parameters,
- `target_projector`: copy of the online projector, with frozen parameters.

Also implement `online_forward(x)` and `target_forward(x)`.

**Hint:**
- Use at least two linear layers in the MLP heads, with a nonlinearity in between. BatchNorm1d is also commonly used in the MLP heads.
- To freeze the target networks, set `requires_grad = False` for all parameters.
- Use `copy.deepcopy` for copying the networks.

In [6]:
importlib.reload(models)
tests.test_byol()

✅ PASS: BYOL class looks correct.


True

---
## Parameter Count
---

The following cell instantiates your BYOL instance and prints the number of parameters.  
It is not an architecture test.

Because BYOL contains online and target networks, the number of trainable parameters should be roughly half of the total number of parameters.

In [7]:
importlib.reload(models)

byol = models.BYOL()
num_parameters = sum(p.numel() for p in byol.parameters())
num_trainable = sum(p.numel() for p in byol.parameters() if p.requires_grad)

print(f"Total parameters: {num_parameters:,}")
print(f"Trainable parameters: {num_trainable:,}")

Total parameters: 2,824,256
Trainable parameters: 1,543,584


---
## BYOL Training Step and Training Loop
---

## **Task:**

Implement the following two functions in `pretrain.py`:

- `byol_training_step`
- `train_byol`

You can copy the structure from previous supervised training loops, but adapt it to BYOL:

- the loader returns two views and an unused CIFAR-10 label,
- compute both BYOL directions:
  - compare the online prediction from the first view with the target projection of the second view,
  - compare the online prediction from the second view with the target projection of the first view,
  - combine the two losses by averaging them.
- backpropagate only through the online network,
- update the target encoder and target projector with EMA after **every** optimizer step.

There are no tests for this section.  
Use the loss curve and downstream accuracy to judge your implementation.

**Hint**: Use `data.make_byol_loader` to create a DataLoader for BYOL pretraining.

---
## Training Setup
---

The code below uses `num_workers=2` as a conservative default. This should work on most machines.  
If training is slow, you can try increasing it. If you get dataloader errors, set it to `0`.

In [8]:
num_workers = 2
batch_size = 256

In [9]:
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "mps"
    if torch.backends.mps.is_available() and torch.backends.mps.is_built() else "cpu")

print(f"Using device: {device}")

Using device: cuda:0


---
## BYOL Pretraining
---

With the provided solution, `20` to `30` epochs is a reasonable reference run.  
For a quick check, start with `5` to `10` epochs.

The BYOL loss should usually decrease clearly during training.  
The absolute value is less important than the downstream result.


In [11]:
importlib.reload(models)
importlib.reload(pretrain)

# Adjust these.
pretrain_lr = 1e-3
pretrain_weight_decay = 1e-4
pretrain_epochs = 30
ema_tau = 0.98

byol, byol_loss = pretrain.train_byol(
    batch_size=batch_size,
    lr=pretrain_lr,
    weight_decay=pretrain_weight_decay,
    epochs=pretrain_epochs,
    device=device,
    tau=ema_tau,
    num_workers=num_workers,
)

visual.show_loss_curve(byol_loss, title="BYOL Pretraining").show()

---
## Collapse Check
---

BYOL is designed to avoid collapse, but implementation mistakes can still produce nearly constant representations.

The helper function below estimates the average standard deviation of encoder features.  
Values extremely close to zero are suspicious.

In [12]:
importlib.reload(data)
importlib.reload(pretrain)

_, test_loader = data.make_cifar10_loaders(batch_size=batch_size, num_workers=num_workers)
feature_std = pretrain.representation_std(
    model=byol,
    data_loader=test_loader,
    device=device,
    max_batches=10,
)

print(f"Mean feature standard deviation: {feature_std:.4f}")

Mean feature standard deviation: 0.8582


---
## Downstream Training
---

Now freeze the BYOL encoder and train a linear classifier on CIFAR-10 labels.

## **Task:**

Implement the linear classifier training code in `downstream.py`.

There are no tests for this section. Reuse the downstream classifier and training code from RotNet or from previous supervised CIFAR-10 exercises.

The provided solution is intended to reach at least **60% CIFAR-10 test accuracy** with `30` BYOL pretraining epochs and `20` linear-classifier epochs.

Possible changes:
- increase the pretraining duration,
- tune the projection and prediction MLP dimensions and architectures,
- tune the optimizer.

**Hint:** Use `data.make_cifar10_loaders` to create DataLoaders for downstream training and testing.

In [14]:
importlib.reload(models)
importlib.reload(downstream)

# Adjust these if you want to improve the result.
downstream_lr = 1e-2
downstream_weight_decay = 1e-4
downstream_epochs = 20

model, train_loss, test_acc = downstream.train_downstream(
    byol=byol,
    batch_size=batch_size,
    lr=downstream_lr,
    weight_decay=downstream_weight_decay,
    epochs=downstream_epochs,
    device=device,
    num_workers=num_workers,
)

visual.show_training_stats(train_loss, test_acc, title="Downstream Training").show()